# Probe Dialects Quality Audit

This notebook audits `data/Finalized/probe_dialects.jsonl` against the finalized annotation file and the original 100-sample source datasets. It is read-only: it reports coverage, format issues, and source-consistency issues without modifying data.

When a new annotation version arrives, change `FINALIZED_JSON` in the setup cell, regenerate `probe_dialects.jsonl`, then run all cells.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
from typing import Any
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path('..') if Path('../data').exists() else Path('.')
FINALIZED_JSON = ROOT / 'data' / 'vialectbench_finalized_6.json'
PROBE_JSONL = ROOT / 'data' / 'Finalized' / 'probe_dialects.jsonl'

SOURCE_PATHS = {
    'SENT': ROOT / 'data' / 'sentiment_100.jsonl',
    'NLI': ROOT / 'data' / 'nli_100.jsonl',
    'QA': ROOT / 'data' / 'qa_100.jsonl',
    'MCQA': ROOT / 'data' / 'mcqa_100.jsonl',
}

TASK_MAP = {'SENT': 'sentiment', 'NLI': 'nli', 'QA': 'qa', 'MCQA': 'mcqa'}
REVERSE_TASK_MAP = {v: k for k, v in TASK_MAP.items()}
DIALECTS = ['PNN', 'PNT1', 'PNT2', 'PNT3', 'PNT4', 'PNB']
EXPECTED_VARIANTS = ['standard'] + DIALECTS
TASK_ORDER = ['MCQA', 'NLI', 'QA', 'SENT']
NLI_LABELS = {'entailment', 'neutral', 'contradiction'}
MCQA_LABELS = set('ABCD')
OPTION_RE = re.compile(r'(?m)^\s*([A-D])\.\s*(.+?)\s*$')
SAMPLE_ID_RE = re.compile(r'^(SENT|NLI|QA|MCQA)_(\d+)(?:_(\d+))?$')

In [ ]:
def load_json(path: Path) -> list[dict[str, Any]]:
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list):
        raise ValueError(f'Expected a JSON list: {path}')
    return data


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize('NFC', str(value or ''))
    return re.sub(r'\s+', ' ', text).strip()


def normalize_nli_label(label: Any) -> str:
    value = str(label or '').strip().lower()
    return {'entailment': 'entailment', 'neutral': 'neutral', 'contradiction': 'contradiction', 'contradict': 'contradiction'}.get(value, value)


def parse_sample_id(sample_id: str) -> tuple[str, int, int | None]:
    match = SAMPLE_ID_RE.fullmatch(str(sample_id))
    if not match:
        raise ValueError(f'Unsupported sample_id format: {sample_id}')
    task, source_index, question_index = match.groups()
    return task, int(source_index), int(question_index) if question_index else None


def parse_mcqa(text: Any) -> tuple[str, list[str]]:
    text = str(text or '')
    matches = list(OPTION_RE.finditer(text))
    if not matches:
        return text.strip(), []
    question = text[:matches[0].start()].strip()
    options_by_label = {match.group(1): match.group(2).strip() for match in matches}
    return question, [options_by_label.get(label, '') for label in 'ABCD']


def mcqa_text(question: Any, options: Any) -> str:
    options = list(options or [])
    option_lines = [f'{label}. {text}' for label, text in zip('ABCD', options)]
    return normalize_text(str(question or '') + '\n' + '\n'.join(option_lines))


def source_maps() -> dict[str, dict[Any, dict[str, Any]]]:
    sentiment = {int(row['_source_index']): row for row in load_jsonl(SOURCE_PATHS['SENT'])}
    nli = {int(row['_source_index']): row for row in load_jsonl(SOURCE_PATHS['NLI'])}
    qa = {int(row['_source_index']): row for row in load_jsonl(SOURCE_PATHS['QA'])}
    mcqa = {(int(row['_source_index']), int(row['_question_index'])): row for row in load_jsonl(SOURCE_PATHS['MCQA'])}
    return {'SENT': sentiment, 'NLI': nli, 'QA': qa, 'MCQA': mcqa}


def source_key_from_sample_id(sample_id: str) -> tuple[str, Any]:
    task_code, source_index, question_index = parse_sample_id(sample_id)
    key = (source_index, question_index) if task_code == 'MCQA' else source_index
    return task_code, key


def source_for_sample(sample_id: str, sources: dict[str, dict[Any, dict[str, Any]]]) -> dict[str, Any] | None:
    task_code, key = source_key_from_sample_id(sample_id)
    return sources.get(task_code, {}).get(key)


def is_empty_variant(task: str, variant: dict[str, Any]) -> bool:
    if task == 'sentiment':
        return normalize_text(variant.get('text')) == ''
    if task == 'nli':
        return normalize_text(variant.get('hypothesis')) == ''
    if task == 'qa':
        return normalize_text(variant.get('question')) == ''
    if task == 'mcqa':
        return normalize_text(variant.get('question')) == '' and len(variant.get('options') or []) == 0
    return True


def add_issue(issues: list[dict[str, Any]], severity: str, check: str, message: str, sample_id: str = '', task: str = '', variant: str = '') -> None:
    issues.append({'severity': severity, 'check': check, 'id': sample_id, 'task': task, 'variant': variant, 'message': message})

## Load Data

In [ ]:
finalized_rows = load_json(FINALIZED_JSON)
probe_cases = load_jsonl(PROBE_JSONL)
sources = source_maps()

finalized_df = pd.DataFrame(finalized_rows)
probe_df = pd.DataFrame([
    {
        'id': case.get('id'),
        'task': case.get('task'),
        'task_code': REVERSE_TASK_MAP.get(case.get('task'), str(case.get('task', '')).upper()),
        'source_index': case.get('source_index'),
        'question_index': case.get('question_index'),
        'annotated_dialects': tuple(case.get('annotated_dialects') or []),
        'num_dialects': case.get('num_dialects'),
        'variant_keys': tuple((case.get('variants') or {}).keys()),
    }
    for case in probe_cases
])

print(f'Finalized rows: {len(finalized_rows):,}')
print(f'Probe cases: {len(probe_cases):,}')
print(f'Source sizes: ' + ', '.join(f'{task}={len(rows):,}' for task, rows in sources.items()))
display(probe_df.head())

## Coverage And Count Checks

A probe case should exist for every finalized sample that has at least one valid dialect row. The probe adds exactly one `standard` variant per case, while dialect rows remain per `sample_id + target_dialect`.

In [ ]:
coverage_issues = []

finalized_df['task_code'] = finalized_df['task'].astype(str).str.upper()
finalized_df['target_dialect'] = finalized_df['target_dialect'].astype(str).str.strip()
valid_dialect_df = finalized_df[finalized_df['target_dialect'].isin(DIALECTS)].copy()

expected_probe_ids = set(valid_dialect_df['sample_id'].astype(str))
actual_probe_ids = set(probe_df['id'].astype(str)) if not probe_df.empty else set()

missing_probe_ids = sorted(expected_probe_ids - actual_probe_ids)
extra_probe_ids = sorted(actual_probe_ids - expected_probe_ids)

for sample_id in missing_probe_ids:
    add_issue(coverage_issues, 'error', 'probe_missing_case', 'Finalized sample has at least one dialect row but no probe case.', sample_id=sample_id)
for sample_id in extra_probe_ids:
    add_issue(coverage_issues, 'error', 'probe_extra_case', 'Probe case does not appear in finalized valid dialect rows.', sample_id=sample_id)

duplicate_pairs = (
    valid_dialect_df.groupby(['sample_id', 'target_dialect']).size().reset_index(name='rows')
    .query('rows > 1')
    .sort_values(['rows', 'sample_id', 'target_dialect'], ascending=[False, True, True])
)
for row in duplicate_pairs.itertuples(index=False):
    add_issue(coverage_issues, 'warning', 'duplicate_finalized_sample_dialect', f'{row.rows} finalized rows for the same sample+dialect; prepare script keeps the last one.', sample_id=row.sample_id, variant=row.target_dialect)

summary_rows = []
for task_code in TASK_ORDER:
    f_task = valid_dialect_df[valid_dialect_df['task_code'] == task_code]
    p_task = probe_df[probe_df['task_code'] == task_code] if not probe_df.empty else pd.DataFrame()
    summary_rows.append({
        'task': task_code,
        'finalized_rows': len(f_task),
        'finalized_unique_samples': f_task['sample_id'].nunique(),
        'finalized_unique_sample_dialect_pairs': f_task[['sample_id', 'target_dialect']].drop_duplicates().shape[0],
        'probe_cases': len(p_task),
        'probe_standard_variants': sum('standard' in keys for keys in p_task.get('variant_keys', [])),
    })

coverage_summary = pd.DataFrame(summary_rows)
display(coverage_summary)
display(pd.DataFrame(coverage_issues).head(50) if coverage_issues else pd.DataFrame([{'status': 'OK', 'message': 'No coverage/count issues found.'}]))
display(duplicate_pairs.head(30))

## Probe Format Checks

Expected task structure:

- MCQA: `standard` has shared `context`, `question`, and exactly 4 `options`; each filled dialect has paraphrased `question` and exactly 4 `options`.
- NLI: `standard` has shared `premise` and original `hypothesis`; each filled dialect has only paraphrased `hypothesis`.
- QA: `standard` has shared `context` and original `question`; each filled dialect has only paraphrased `question`.
- Sentiment: `standard` and filled dialects each have `text`.

Empty dialect slots are allowed only when the dialect is not listed in `annotated_dialects`.

In [ ]:
format_issues = []

def check_variant_format(case: dict[str, Any]) -> None:
    sample_id = str(case.get('id', ''))
    task = str(case.get('task', ''))
    task_code = REVERSE_TASK_MAP.get(task, task.upper())
    variants = case.get('variants') or {}
    annotated = set(case.get('annotated_dialects') or [])

    if task not in REVERSE_TASK_MAP:
        add_issue(format_issues, 'error', 'unknown_task', f'Unsupported probe task: {task}', sample_id, task)
        return
    if set(variants.keys()) != set(EXPECTED_VARIANTS):
        missing = sorted(set(EXPECTED_VARIANTS) - set(variants.keys()))
        extra = sorted(set(variants.keys()) - set(EXPECTED_VARIANTS))
        add_issue(format_issues, 'error', 'variant_keys', f'Missing variants={missing}; extra variants={extra}', sample_id, task)
    if sorted(annotated) != sorted([d for d in DIALECTS if not is_empty_variant(task, variants.get(d, {}))]):
        add_issue(format_issues, 'warning', 'annotated_dialects_mismatch', 'annotated_dialects differs from non-empty dialect slots.', sample_id, task)
    if case.get('num_dialects') != len(annotated):
        add_issue(format_issues, 'warning', 'num_dialects_mismatch', f'num_dialects={case.get("num_dialects")} but annotated_dialects has {len(annotated)}.', sample_id, task)

    standard = variants.get('standard') or {}
    if task == 'mcqa':
        if normalize_text(standard.get('context')) == '' or normalize_text(standard.get('question')) == '':
            add_issue(format_issues, 'error', 'mcqa_standard_fields', 'MCQA standard must have non-empty context and question.', sample_id, task, 'standard')
        if len(standard.get('options') or []) != 4 or any(normalize_text(x) == '' for x in standard.get('options') or []):
            add_issue(format_issues, 'error', 'mcqa_standard_options', 'MCQA standard must have exactly 4 non-empty options.', sample_id, task, 'standard')
        if str(case.get('label', '')).strip().upper() not in MCQA_LABELS:
            add_issue(format_issues, 'error', 'mcqa_label', f'Invalid MCQA label: {case.get("label")}', sample_id, task)
    elif task == 'nli':
        if normalize_text(standard.get('premise')) == '' or normalize_text(standard.get('hypothesis')) == '':
            add_issue(format_issues, 'error', 'nli_standard_fields', 'NLI standard must have non-empty premise and hypothesis.', sample_id, task, 'standard')
        if normalize_nli_label(case.get('label')) not in NLI_LABELS:
            add_issue(format_issues, 'error', 'nli_label', f'Invalid NLI label: {case.get("label")}', sample_id, task)
    elif task == 'qa':
        if normalize_text(standard.get('context')) == '' or normalize_text(standard.get('question')) == '':
            add_issue(format_issues, 'error', 'qa_standard_fields', 'QA standard must have non-empty context and question.', sample_id, task, 'standard')
        if not case.get('answers'):
            add_issue(format_issues, 'error', 'qa_answers', 'QA case must have non-empty answers.', sample_id, task)
    elif task == 'sentiment':
        if normalize_text(standard.get('text')) == '':
            add_issue(format_issues, 'error', 'sentiment_standard_text', 'Sentiment standard text is empty.', sample_id, task, 'standard')
        if normalize_text(case.get('label')) == '':
            add_issue(format_issues, 'error', 'sentiment_label', 'Sentiment label is empty.', sample_id, task)

    for dialect in DIALECTS:
        variant = variants.get(dialect) or {}
        empty = is_empty_variant(task, variant)
        if dialect in annotated and empty:
            add_issue(format_issues, 'error', 'filled_dialect_empty', 'Dialect is listed as annotated but variant is empty.', sample_id, task, dialect)
        if dialect not in annotated and not empty:
            add_issue(format_issues, 'warning', 'unannotated_dialect_non_empty', 'Dialect variant is non-empty but dialect is not listed as annotated.', sample_id, task, dialect)

        if dialect in annotated:
            if task == 'mcqa':
                if normalize_text(variant.get('question')) == '' or len(variant.get('options') or []) != 4 or any(normalize_text(x) == '' for x in variant.get('options') or []):
                    add_issue(format_issues, 'error', 'mcqa_dialect_options', 'Filled MCQA dialect must have a non-empty question and exactly 4 non-empty options.', sample_id, task, dialect)
                if 'context' in variant:
                    add_issue(format_issues, 'warning', 'mcqa_dialect_context', 'MCQA dialect variant should not carry context; context is shared from standard.', sample_id, task, dialect)
            elif task == 'nli':
                if normalize_text(variant.get('hypothesis')) == '':
                    add_issue(format_issues, 'error', 'nli_dialect_hypothesis', 'Filled NLI dialect must have non-empty hypothesis.', sample_id, task, dialect)
                if 'premise' in variant:
                    add_issue(format_issues, 'warning', 'nli_dialect_premise', 'NLI dialect variant should not carry premise; premise is shared from standard.', sample_id, task, dialect)
            elif task == 'qa':
                if normalize_text(variant.get('question')) == '':
                    add_issue(format_issues, 'error', 'qa_dialect_question', 'Filled QA dialect must have non-empty question.', sample_id, task, dialect)
                if 'context' in variant:
                    add_issue(format_issues, 'warning', 'qa_dialect_context', 'QA dialect variant should not carry context; context is shared from standard.', sample_id, task, dialect)
            elif task == 'sentiment':
                if normalize_text(variant.get('text')) == '':
                    add_issue(format_issues, 'error', 'sentiment_dialect_text', 'Filled sentiment dialect must have non-empty text.', sample_id, task, dialect)


for case in probe_cases:
    check_variant_format(case)

format_issues_df = pd.DataFrame(format_issues)
if format_issues_df.empty:
    display(pd.DataFrame([{'status': 'OK', 'message': 'No probe format issues found.'}]))
else:
    display(format_issues_df.groupby(['severity', 'check']).size().reset_index(name='n').sort_values(['severity', 'n'], ascending=[True, False]))
    display(format_issues_df.head(100))

## Finalized Rows Versus Original 100-Sample Files

This validates every finalized annotation row against the original task source file before conversion to probe format.

In [ ]:
source_issues = []

def validate_finalized_row_against_source(row: dict[str, Any]) -> None:
    sample_id = str(row.get('sample_id', ''))
    try:
        task_code, _, _ = parse_sample_id(sample_id)
    except ValueError as exc:
        add_issue(source_issues, 'error', 'sample_id_format', str(exc), sample_id=sample_id)
        return
    source = source_for_sample(sample_id, sources)
    if source is None:
        add_issue(source_issues, 'error', 'source_missing', 'Sample id was not found in the original 100-sample source files.', sample_id=sample_id, task=task_code)
        return

    if task_code == 'SENT':
        if normalize_text(row.get('original_text')) != normalize_text(source.get('Sentence')):
            add_issue(source_issues, 'error', 'sentiment_original_text_mismatch', 'original_text differs from source Sentence.', sample_id, task_code)
        if normalize_text(row.get('label')) != normalize_text(source.get('Emotion')):
            add_issue(source_issues, 'error', 'sentiment_label_mismatch', 'label differs from source Emotion.', sample_id, task_code)
    elif task_code == 'NLI':
        if normalize_text(row.get('original_text')) != normalize_text(source.get('premise')):
            add_issue(source_issues, 'error', 'nli_premise_mismatch', 'original_text differs from source premise.', sample_id, task_code)
        if normalize_text(row.get('hypothesis')) != normalize_text(source.get('hypothesis')):
            add_issue(source_issues, 'error', 'nli_hypothesis_mismatch', 'hypothesis differs from source hypothesis.', sample_id, task_code)
        if normalize_nli_label(row.get('label')) != normalize_nli_label(source.get('label')):
            add_issue(source_issues, 'error', 'nli_label_mismatch', 'label differs from source label.', sample_id, task_code)
    elif task_code == 'QA':
        if normalize_text(row.get('original_text')) != normalize_text(source.get('question')):
            add_issue(source_issues, 'error', 'qa_question_mismatch', 'original_text differs from source question.', sample_id, task_code)
        source_answers = [normalize_text(x) for x in (source.get('answers') or {}).get('text', [])]
        row_answer = normalize_text(row.get('label') or row.get('hypothesis'))
        if row_answer and row_answer not in source_answers:
            add_issue(source_issues, 'error', 'qa_answer_mismatch', 'label/hypothesis answer is not in source answers.', sample_id, task_code)
    elif task_code == 'MCQA':
        question, options = parse_mcqa(row.get('original_text'))
        if mcqa_text(question, options) != mcqa_text(source.get('question'), source.get('options')):
            add_issue(source_issues, 'error', 'mcqa_question_options_mismatch', 'original_text question/options differ from source question/options.', sample_id, task_code)
        label = normalize_text(row.get('label')).upper()
        if label != normalize_text(source.get('answer')).upper():
            add_issue(source_issues, 'error', 'mcqa_answer_label_mismatch', 'label differs from source answer.', sample_id, task_code)
        answer_index = 'ABCD'.find(label)
        source_options = source.get('options') or []
        if 0 <= answer_index < len(source_options) and normalize_text(row.get('hypothesis')) != normalize_text(source_options[answer_index]):
            add_issue(source_issues, 'error', 'mcqa_answer_text_mismatch', 'hypothesis differs from the labeled source option.', sample_id, task_code)


for row in finalized_rows:
    validate_finalized_row_against_source(row)

source_issues_df = pd.DataFrame(source_issues)
if source_issues_df.empty:
    display(pd.DataFrame([{'status': 'OK', 'message': 'All finalized rows are consistent with source 100-sample files.'}]))
else:
    display(source_issues_df.groupby(['severity', 'check']).size().reset_index(name='n').sort_values(['n'], ascending=False))
    display(source_issues_df.head(100))

## Probe Cases Versus Source And Finalized Rows

This validates the generated probe file itself: standard variants should match original sources, and dialect variants should match the finalized annotation rows that produced them. For duplicate finalized `sample_id + dialect` rows, the conversion script keeps the last row, so this audit compares against the last row in file order.

In [ ]:
probe_consistency_issues = []

last_finalized_by_pair = {}
for row in finalized_rows:
    dialect = str(row.get('target_dialect', '')).strip()
    if dialect in DIALECTS:
        last_finalized_by_pair[(str(row.get('sample_id')), dialect)] = row


def validate_probe_standard(case: dict[str, Any]) -> None:
    sample_id = str(case.get('id', ''))
    task = str(case.get('task', ''))
    task_code = REVERSE_TASK_MAP.get(task, task.upper())
    source = source_for_sample(sample_id, sources)
    standard = (case.get('variants') or {}).get('standard') or {}
    if source is None:
        add_issue(probe_consistency_issues, 'error', 'probe_source_missing', 'Probe case id not found in source files.', sample_id, task_code, 'standard')
        return
    if task == 'sentiment':
        if normalize_text(standard.get('text')) != normalize_text(source.get('Sentence')):
            add_issue(probe_consistency_issues, 'error', 'probe_sentiment_standard_mismatch', 'Probe standard text differs from source Sentence.', sample_id, task_code, 'standard')
        if normalize_text(case.get('label')) != normalize_text(source.get('Emotion')):
            add_issue(probe_consistency_issues, 'error', 'probe_sentiment_label_mismatch', 'Probe label differs from source Emotion.', sample_id, task_code)
    elif task == 'nli':
        if normalize_text(standard.get('premise')) != normalize_text(source.get('premise')):
            add_issue(probe_consistency_issues, 'error', 'probe_nli_premise_mismatch', 'Probe standard premise differs from source premise.', sample_id, task_code, 'standard')
        if normalize_text(standard.get('hypothesis')) != normalize_text(source.get('hypothesis')):
            add_issue(probe_consistency_issues, 'error', 'probe_nli_hypothesis_mismatch', 'Probe standard hypothesis differs from source hypothesis.', sample_id, task_code, 'standard')
        if normalize_nli_label(case.get('label')) != normalize_nli_label(source.get('label')):
            add_issue(probe_consistency_issues, 'error', 'probe_nli_label_mismatch', 'Probe label differs from source label.', sample_id, task_code)
    elif task == 'qa':
        if normalize_text(standard.get('context')) != normalize_text(source.get('context')):
            add_issue(probe_consistency_issues, 'error', 'probe_qa_context_mismatch', 'Probe standard context differs from source context.', sample_id, task_code, 'standard')
        if normalize_text(standard.get('question')) != normalize_text(source.get('question')):
            add_issue(probe_consistency_issues, 'error', 'probe_qa_question_mismatch', 'Probe standard question differs from source question.', sample_id, task_code, 'standard')
        source_answers = [normalize_text(x) for x in (source.get('answers') or {}).get('text', [])]
        probe_answers = [normalize_text(x) for x in (case.get('answers') or [])]
        if source_answers and probe_answers != source_answers:
            add_issue(probe_consistency_issues, 'error', 'probe_qa_answers_mismatch', 'Probe answers differ from source answers.', sample_id, task_code)
    elif task == 'mcqa':
        if normalize_text(standard.get('context')) != normalize_text(source.get('article')):
            add_issue(probe_consistency_issues, 'error', 'probe_mcqa_context_mismatch', 'Probe standard context differs from source article.', sample_id, task_code, 'standard')
        if mcqa_text(standard.get('question'), standard.get('options')) != mcqa_text(source.get('question'), source.get('options')):
            add_issue(probe_consistency_issues, 'error', 'probe_mcqa_question_options_mismatch', 'Probe standard question/options differ from source question/options.', sample_id, task_code, 'standard')
        if normalize_text(case.get('label')).upper() != normalize_text(source.get('answer')).upper():
            add_issue(probe_consistency_issues, 'error', 'probe_mcqa_label_mismatch', 'Probe label differs from source answer.', sample_id, task_code)


def validate_probe_dialects_against_finalized(case: dict[str, Any]) -> None:
    sample_id = str(case.get('id', ''))
    task = str(case.get('task', ''))
    task_code = REVERSE_TASK_MAP.get(task, task.upper())
    variants = case.get('variants') or {}
    annotated = set(case.get('annotated_dialects') or [])
    for dialect in DIALECTS:
        row = last_finalized_by_pair.get((sample_id, dialect))
        variant = variants.get(dialect) or {}
        if row is None:
            if dialect in annotated or not is_empty_variant(task, variant):
                add_issue(probe_consistency_issues, 'error', 'probe_dialect_without_finalized_row', 'Probe dialect is filled/listed but no finalized row exists.', sample_id, task_code, dialect)
            continue
        if dialect not in annotated:
            add_issue(probe_consistency_issues, 'warning', 'finalized_row_not_in_annotated_dialects', 'Finalized row exists but dialect is not listed in annotated_dialects.', sample_id, task_code, dialect)
        if task == 'sentiment':
            if normalize_text(variant.get('text')) != normalize_text(row.get('dialect_text')):
                add_issue(probe_consistency_issues, 'error', 'probe_sentiment_dialect_mismatch', 'Probe dialect text differs from finalized dialect_text.', sample_id, task_code, dialect)
        elif task == 'nli':
            if normalize_text(variant.get('hypothesis')) != normalize_text(row.get('dialect_text')):
                add_issue(probe_consistency_issues, 'error', 'probe_nli_dialect_mismatch', 'Probe dialect hypothesis differs from finalized dialect_text.', sample_id, task_code, dialect)
        elif task == 'qa':
            if normalize_text(variant.get('question')) != normalize_text(row.get('dialect_text')):
                add_issue(probe_consistency_issues, 'error', 'probe_qa_dialect_mismatch', 'Probe dialect question differs from finalized dialect_text.', sample_id, task_code, dialect)
        elif task == 'mcqa':
            question, options = parse_mcqa(row.get('dialect_text'))
            if mcqa_text(variant.get('question'), variant.get('options')) != mcqa_text(question, options):
                add_issue(probe_consistency_issues, 'error', 'probe_mcqa_dialect_mismatch', 'Probe dialect question/options differ from parsed finalized dialect_text.', sample_id, task_code, dialect)


for case in probe_cases:
    validate_probe_standard(case)
    validate_probe_dialects_against_finalized(case)

probe_consistency_issues_df = pd.DataFrame(probe_consistency_issues)
if probe_consistency_issues_df.empty:
    display(pd.DataFrame([{'status': 'OK', 'message': 'Probe cases are consistent with sources and finalized rows.'}]))
else:
    display(probe_consistency_issues_df.groupby(['severity', 'check']).size().reset_index(name='n').sort_values(['n'], ascending=False))
    display(probe_consistency_issues_df.head(100))

## Compact QA Views

Use these tables for quick manual inspection of suspected rows.

In [ ]:
all_issues = pd.concat(
    [df for df in [pd.DataFrame(coverage_issues), format_issues_df, source_issues_df, probe_consistency_issues_df] if not df.empty],
    ignore_index=True,
) if any(not df.empty for df in [pd.DataFrame(coverage_issues), format_issues_df, source_issues_df, probe_consistency_issues_df]) else pd.DataFrame(columns=['severity', 'check', 'id', 'task', 'variant', 'message'])

if all_issues.empty:
    display(pd.DataFrame([{'status': 'OK', 'message': 'No issues found by automated checks.'}]))
else:
    display(all_issues.groupby(['severity', 'check']).size().reset_index(name='n').sort_values(['severity', 'n'], ascending=[True, False]))
    display(all_issues.sort_values(['severity', 'check', 'id', 'variant']).head(200))

dialect_fill_rows = []
for case in probe_cases:
    task = case.get('task')
    variants = case.get('variants') or {}
    for dialect in DIALECTS:
        dialect_fill_rows.append({
            'task': REVERSE_TASK_MAP.get(task, task),
            'id': case.get('id'),
            'dialect': dialect,
            'filled': not is_empty_variant(task, variants.get(dialect, {})),
            'listed_annotated': dialect in set(case.get('annotated_dialects') or []),
        })
dialect_fill_df = pd.DataFrame(dialect_fill_rows)
display(dialect_fill_df.groupby(['task', 'dialect'])['filled'].agg(['sum', 'count']).reset_index().rename(columns={'sum': 'filled_cases', 'count': 'probe_cases'}))

## MCQA Option Error Attribution

This cell separates MCQA dialect option errors caused by a bad Standard/source case from dialect-specific formatting errors. If dialect errors only occur inside samples whose Standard variant is already missing options, then the issue is sample-level rather than variant-specific.


In [ ]:
mcqa_option_rows = []
for case in probe_cases:
    if case.get('task') != 'mcqa':
        continue
    sample_id = case.get('id')
    variants = case.get('variants') or {}
    standard = variants.get('standard') or {}
    standard_options = standard.get('options') or []
    standard_bad = len(standard_options) != 4 or any(normalize_text(option) == '' for option in standard_options)

    for variant_name in EXPECTED_VARIANTS:
        variant = variants.get(variant_name) or {}
        if variant_name != 'standard' and variant_name not in set(case.get('annotated_dialects') or []):
            continue
        options = variant.get('options') or []
        option_count = len(options)
        empty_option_count = sum(normalize_text(option) == '' for option in options)
        bad_options = option_count != 4 or empty_option_count > 0
        mcqa_option_rows.append({
            'id': sample_id,
            'variant': variant_name,
            'is_standard': variant_name == 'standard',
            'standard_bad': standard_bad,
            'bad_options': bad_options,
            'option_count': option_count,
            'empty_option_count': empty_option_count,
            'options': options,
        })

mcqa_option_df = pd.DataFrame(mcqa_option_rows)
standard_bad_ids = set(
    mcqa_option_df.query('is_standard and bad_options')['id']
)
dialect_bad_df = mcqa_option_df.query('not is_standard and bad_options').copy()
dialect_bad_from_bad_standard = dialect_bad_df[dialect_bad_df['id'].isin(standard_bad_ids)]
dialect_bad_from_good_standard = dialect_bad_df[~dialect_bad_df['id'].isin(standard_bad_ids)]

summary = pd.DataFrame([
    {
        'metric': 'MCQA samples with bad standard options',
        'count': len(standard_bad_ids),
    },
    {
        'metric': 'Bad dialect variants whose sample standard is also bad',
        'count': len(dialect_bad_from_bad_standard),
    },
    {
        'metric': 'Unique samples in previous row',
        'count': dialect_bad_from_bad_standard['id'].nunique(),
    },
    {
        'metric': 'Bad dialect variants whose sample standard is valid',
        'count': len(dialect_bad_from_good_standard),
    },
    {
        'metric': 'Unique samples in previous row',
        'count': dialect_bad_from_good_standard['id'].nunique(),
    },
])
display(summary)

bad_standard_variant_counts = (
    dialect_bad_from_bad_standard
    .groupby('id')['variant']
    .apply(lambda values: ', '.join(values))
    .reset_index(name='bad_dialect_variants')
    .merge(
        mcqa_option_df.query('is_standard')[['id', 'option_count', 'empty_option_count', 'options']],
        on='id',
        how='left',
    )
    .sort_values('id')
)
print('Dialect option errors inside samples whose Standard is already bad:')
display(bad_standard_variant_counts)

print('Dialect option errors where Standard has 4 non-empty options:')
if dialect_bad_from_good_standard.empty:
    display(pd.DataFrame([{'status': 'OK', 'message': 'No dialect-specific MCQA option errors after excluding bad-standard samples.'}]))
else:
    display(
        dialect_bad_from_good_standard
        .groupby('id')['variant']
        .apply(lambda values: ', '.join(values))
        .reset_index(name='bad_dialect_variants')
        .sort_values('id')
    )
